# 06. Model A Assembly (최종 구조화 문서 생성)

02~05에서 채운 필드들을 합쳐 Model A의 최종 출력을 만듭니다.

**입력:** `outputs/table_recognition/batch_table.json`  
**출력:**
- `outputs/model_a/batch_model_a.json` — Model B에 전달할 구조화 JSON
- `outputs/model_a/page_XX_readable.txt` — 읽기 순서대로 정렬된 텍스트 (검수용)
- `outputs/model_a/completeness_report.json` — 필드 완성도 리포트

In [ ]:
import os
import json

TABLE_DIR  = "../outputs/table_recognition"
OUTPUT_DIR = "../outputs/model_a"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
with open(os.path.join(TABLE_DIR, "batch_table.json"), encoding="utf-8") as f:
    all_pages = json.load(f)

print(f"총 {len(all_pages)}페이지 로드")

## 1. 완성도 검사

각 요소 타입별로 필드가 얼마나 채워졌는지 확인합니다.

In [ ]:
TEXT_TYPES    = {"title", "sectionheader", "text", "listitem", "caption", "pageheader", "pagefooter", "handwriting"}
FORMULA_TYPES = {"equation"}
TABLE_TYPES   = {"table"}
IMAGE_TYPES   = {"picture", "figure"}  # Model B 담당

stats = {
    "total": 0,
    "text_filled": 0, "text_total": 0,
    "latex_filled": 0, "latex_total": 0,
    "table_filled": 0, "table_total": 0,
    "image_pending": 0,
    "unknown_types": set()
}

for page in all_pages:
    for e in page["elements"]:
        stats["total"] += 1
        t = e["type"]
        if t in TEXT_TYPES:
            stats["text_total"] += 1
            if e.get("text"):
                stats["text_filled"] += 1
        elif t in FORMULA_TYPES:
            stats["latex_total"] += 1
            if e.get("latex"):
                stats["latex_filled"] += 1
        elif t in TABLE_TYPES:
            stats["table_total"] += 1
            if e.get("text"):
                stats["table_filled"] += 1
        elif t in IMAGE_TYPES:
            stats["image_pending"] += 1
        else:
            stats["unknown_types"].add(t)

stats["unknown_types"] = list(stats["unknown_types"])

print(f"전체 요소: {stats['total']}개")
print(f"텍스트:   {stats['text_filled']}/{stats['text_total']} 완성")
print(f"수식:     {stats['latex_filled']}/{stats['latex_total']} 완성")
print(f"표:       {stats['table_filled']}/{stats['table_total']} 완성")
print(f"이미지:   {stats['image_pending']}개 → Model B 대기")
if stats["unknown_types"]:
    print(f"미분류 타입: {stats['unknown_types']}")

## 2. 최종 구조화 JSON 생성

각 요소에 `content` 필드를 추가합니다.
- 텍스트 → `text`
- 수식 → `latex`
- 표 → `text` (Markdown)
- 이미지 → `null` (Model B가 채울 예정)

In [ ]:
def get_content(element):
    t = element["type"]
    if t in TEXT_TYPES | TABLE_TYPES:
        return element.get("text")
    elif t in FORMULA_TYPES:
        latex = element.get("latex")
        return f"$${latex}$$" if latex and not latex.startswith("ERROR") else latex
    elif t in IMAGE_TYPES:
        return None  # Model B 대기
    return element.get("text")


for page in all_pages:
    # 읽기 순서 정렬
    elements = sorted(page["elements"], key=lambda e: e.get("reading_order", 9999))
    for elem in elements:
        elem["content"] = get_content(elem)
        elem["model_b_required"] = elem["type"] in IMAGE_TYPES

print("content 필드 생성 완료")

## 3. 검수용 텍스트 출력

읽기 순서대로 내용을 이어붙여 사람이 읽을 수 있는 형태로 출력합니다.

In [ ]:
def page_to_readable(page):
    lines = [f"{'='*50}", f"[ {page['page_id']}페이지 ]\n"]
    elements = sorted(page["elements"], key=lambda e: e.get("reading_order", 9999))
    for elem in elements:
        t = elem["type"]
        content = elem.get("content")
        if t in IMAGE_TYPES:
            lines.append(f"[이미지 — Model B 대기: {elem['id']}]")
        elif content:
            prefix = {"title": "# ", "sectionheader": "## "}.get(t, "")
            lines.append(prefix + content)
        else:
            lines.append(f"[{t} — 내용 없음: {elem['id']}]")
        lines.append("")
    return "\n".join(lines)


# 모든 페이지 텍스트 파일 저장
for page in all_pages:
    readable = page_to_readable(page)
    out_path = os.path.join(OUTPUT_DIR, f"page_{page['page_id']:02d}_readable.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(readable)

# 첫 페이지 미리보기
print(page_to_readable(all_pages[0]))

## 4. 최종 JSON 저장

In [ ]:
# Model B 전달용 JSON — picture/figure만 추려서 별도 저장
model_b_queue = []
for page in all_pages:
    for elem in page["elements"]:
        if elem.get("model_b_required"):
            model_b_queue.append({
                "page_id": page["page_id"],
                "element_id": elem["id"],
                "type": elem["type"],
                "bbox_px": elem["bbox_px"],
                "bbox_norm": elem["bbox_norm"],
                "reading_order": elem.get("reading_order")
            })

# 저장
model_a_path = os.path.join(OUTPUT_DIR, "batch_model_a.json")
with open(model_a_path, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, ensure_ascii=False, indent=2)

model_b_path = os.path.join(OUTPUT_DIR, "model_b_queue.json")
with open(model_b_path, "w", encoding="utf-8") as f:
    json.dump(model_b_queue, f, ensure_ascii=False, indent=2)

report_path = os.path.join(OUTPUT_DIR, "completeness_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"batch_model_a.json  → {model_a_path}")
print(f"model_b_queue.json  → {model_b_path}  ({len(model_b_queue)}개 이미지 대기)")
print(f"completeness_report → {report_path}")